In [31]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [33]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [75]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python" or "json" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code
* Respond ONLY the JSON Object, no markdown, no backticks, no explanation.
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)

    return json.loads(text)

In [73]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2)

In [76]:
print(dataset)

[{'task': 'Parse an AWS S3 bucket name from a full S3 URI like s3://my-bucket/path/to/file.txt', 'format': 'regex'}, {'task': 'Create a JSON policy document that allows read-only access to a specific S3 bucket', 'format': 'json'}, {'task': 'Write a Python function that validates an AWS access key ID format', 'format': 'python'}, {'task': 'Extract the region from an AWS ARN like arn:aws:s3:::my-bucket', 'format': 'regex'}, {'task': 'Create a JSON CloudFormation template snippet for an S3 bucket with versioning enabled', 'format': 'json'}, {'task': 'Write a Python function that converts an AWS timestamp to a human-readable format', 'format': 'python'}, {'task': 'Match all valid AWS Lambda function names using a regular expression', 'format': 'regex'}, {'task': 'Create a JSON IAM role trust policy for an EC2 instance', 'format': 'json'}, {'task': 'Write a Python function that checks if an AWS account ID is valid', 'format': 'python'}, {'task': 'Extract the service name from an AWS ARN str

In [85]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then return the results"""
  prompt = f"""
  Solve the following task:

  {test_case['task']}

  * Response only with Python, JSON, or a plain Regex; without any markdown, backticks, or explanation.
  * Do not add any comments or commentary or explanation.
  """
  
  messages = []
  add_user_message(messages, prompt)
  # add_assistant_message(messages, "```code")
  output = chat(messages)

  return output

In [78]:
def grade_by_model(test_case, output):
  # Create evaluation prompt
  eval_prompt = f"""
  You are an expert code reviewer. Evaluate this AI-generated solution.
  
  Original Task:
  <task>
  {test_case['task']}
  </task>

  Solution to Evaluate: 
  <solution>
  {output}
  </solution>
  
  Output Format:
  Provide your evaluation as a structured JSON object with:
  - "strengths": An array of 1-3 key strengths
  - "weaknesses": An array of 1-3 key areas for improvement  
  - "reasoning": A concise explanation of your assessment
  - "score": A number between 1-10
  Respond ONLY with the JSON, no markdown, no backticks, no explanation.

  Keep your response concise and direct.
  Example response shape:
  {{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
  }}
  """

  messages = []
  
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages,"{")
  
  eval_text = chat(messages)

  return json.loads("{" + eval_text)

In [79]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {format}")

In [88]:
import re

def strip_code_blocks(text: str) -> str:
  pattern = r'^```[\w]*\n?(.*?)```$'
  match = re.search(pattern, text.strip(), re.DOTALL)
  if match:
      return match.group(1).strip()
  return text.strip()

def run_test_case(test_case):
  """Calls run_prompt, then grades the results"""
  output = run_prompt(test_case)
  output = strip_code_blocks(output)

  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  reasoning = model_grade["reasoning"]

  syntax_score = grade_syntax(output, test_case)

  score = (model_score + syntax_score) / 2

  return {
    "output": output,
    "test_case": test_case,
    "score": score,
    "reasoning": reasoning
  }

In [81]:
def run_eval(dataset):
  """Loads the dataset and calls run test case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  return results

In [89]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

In [90]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "import re\n\ndef parse_s3_bucket(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    if match:\n        return match.group(1)\n    return None\n\n# Test examples\ntest_uris = [\n    \"s3://my-bucket/path/to/file.txt\",\n    \"s3://another-bucket/\",\n    \"s3://bucket-123/deep/path/structure/file.json\"\n]\n\nfor uri in test_uris:\n    print(f\"{uri} -> {parse_s3_bucket(uri)}\")",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 URI like s3://my-bucket/path/to/file.txt",
      "format": "regex"
    },
    "score": 3.5,
    "reasoning": "The solution correctly solves the core parsing task with a simple, effective regex pattern. The regex properly captures the bucket name between 's3://' and the first slash. However, it lacks defensive programming practices like input validation and AWS-specific bucket name constraints. The code is functional but could be more robust for production use."
  },
  {
    "output": "{\n  \"Version\": \